<br>

<img src="https://lindas.admin.ch/static-assets/img/lindaslogo.png" style="width:15%; float:right">

# Tutorial version.link und historisiertes Gemeindeverzeichnis

## Einführung

Dieses Tutorial ist dazu gedacht, eine Einführung in zwei verschiedene, aber zusammenhängende Themen zu geben:

1. das historisierte Gemeindeverzeichnis
2. das version.link Schema

Der Grund für dies Verknüpfung ist, dass das historisierte Gemeindeverzeichnis das version.link Schema verwendet. Dieses Tutorial zeigt dabei sowohl die grundlegenden Mechanismen des version.link Schemas auf, als auch die Arbeit mit dem historisierten Gemeindeverzeichnis. Das Tutorial ist also von Interesse für Personen, die sich über version.link informieren möchten - und das passiert hier anhand des historisierten Gemeindeverzeichnisses, aber auch für Personen, die sich primär für das historisierte Gemeindeverzeichnis interessieren, dafür aber über grundlegende Kenntnisse von version.link verfügen müssen.

## version.link Schema

Das version.link Schema wurde dazu entworfen, Daten, die in einem hierarchischen Zusammenhang stehen inklusive deren zeitlichen Entwicklung zu modellieren. Die Dokumentation von version.link ist unter https://version.link zu finden.

## Das historisierte Gemeindeverzeichnis

Gemeinden in der Schweiz unterliegen Veränderungen. Aktuell sind das primär Fusionen zwischen einzelnen Gemeinden. Das historisierte Gemeindeverzeichnis gibt sowohl einen Überblick über alle aktuell existierenden Gemeinden als auch über Gemeinden im Verlaufe der Zeit. Im historischen Gemeindeverzeichnis sind also auch solche Gemeinden zu finden, die heute nicht mehr existieren, weil sie fusioniert haben.

## Zielpublikum

Das Zielpublikum dieses Tutorials sind Personen, die über gute Grundkenntnisse in der Informatik verfügen, die aber noch wenig Knowhow zum Thema Linked Data, SPARQL und version.link besitzen.

## Interaktives Notebook

Dieses Tutorial ist ein sogenanntes **interaktives JupyterLite Notebook**. In diesem Notebook können Sie den Inhalt der einzelnen Zellen interaktiv ändern und diese Zellen direkt ausführen, um das Ergebnis Ihrer Änderungen sofort zu sehen. Die Zellen enthalten entweder [Markdown](https://en.wikipedia.org/wiki/Markdown)-Inhalt (wie diese Zelle) oder ausführbaren Python-Quellcode. Dies ist für ein Tutorial sehr hilfreich, weil Inhalte beliebig mit ausführbarem Quellcode kombiniert werden können. Es können also Abfragen gezeigt werden, diese erklärt werden und darauf weiter aufgebaut werden.

**Um direkt loslegen zu können klicken Sie oben im Menu auf Run -> Run All Cells.**  
**Einzelne ausgewählte Zellen können sie danach abändern und mit dem "Play-Button" erneut ausführen und so Abfragen individuell anpassen.**

Das Notebook startet mit einem [Setup](#Setup) der Programierumgebung. Das eigentliche Tutorial startet [hier](#Tutorial).

*Zusätzliche Informationen zu JupyterLite:*  
JupyterLite is ein spezielles Jupyter Notebook mit dem Vorteil, vollständig browserbasiert zu sein, ohne eine aufwändige Backend-Infrastruktur zu benötigen. Der Nachteil ist, dass die erstmalige Ausführung der Zellen einige Zeit in Anspruch nehmen kann, weil eine erhebliche Menge von Daten geladen werden muss. Dass eine Zelle noch in Ausführung ist, ist am `[*]` links neben der Zelle erkennbar. Nach Abschluss der Ausführung erscheint statt eines `*` eine Zahl. Vor der ersten Ausführung ist eine leere Klammer `[ ]` zu sehen. Nachfolgende wiederholte Ausführungen von Zellen werden aufgrund der gespeicherten Daten in Ihrem Browser-Cache viel schneller sein. Wenn Sie Änderungen am Tutorial vornehmen, werden diese Änderungen automatisch lokal im Browser-Cache gespeichert. Bei einem darauffolgenden Öffnen werden wiederum die Änderungen aus dem Browser-Cache geladen. Um wieder zur Ursprüngsversion des Tutorials zurückzukehren, müssen entweder die Browser-Daten gelöscht werden, oder Sie öffnen das Tutorial in einem Chrome-Inkognito Fenster, wo die Änderungen nach Schliessen des Fensters aus dem Cache gelöscht werden.

Wenn Sie mehr über die Handhabung von Jupyter Notebooks wissen wollen, finden Sie hier zwei nützliche Ressourcen:

- [Die JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [Das Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

## Setup

Eine SPARQL Abfrage ist nichts anderes als ein POST-Request an den entsprechenden Triple Store Datenbank Server. Um diese Requests und die erhaltenen Antworten einfacher handhaben zu können, enthält dieses Notebook vorbereiteten Python Code, der nachfolgend importiert wird. Zusätzlich wird das `pandas` Modul importiert, welches die Möglichkeit bietet, die tabellarischen Daten der SPARQL Abfragen als [Pandas Dataframes](https://pandas.pydata.org/docs/index.html) weiter zu verarbeiten. 

In [1]:
import pandas as pd
from ext.sparql import query, display_result

# Linked Data Einführung

Linked Data beschreibt ein **Framework für den Umgang mit Daten**, die sowohl für Menschen nützlich sein sollen, als auch maschinenlesbar sind inkl. einer von Computern verarbeitbaren Semantik. Also sowohl Menschen als auch Computer sollen die Daten "verstehen" und interpretieren können. 

## RDF

Das für Linked Data verwendete Datenformat ist RDF (Resource Description Framework). Das bedeutet, dass die Daten nicht als Tabellen (wie beispielsweise in relationalen Datenbanken) sondern als **Triples** abgespeichert werden. Triples folgen der grammatikalischen Struktur **Subjekt -> Prädikat -> Objekt** und können auch als grammatikalischer Satz verstanden werden. 

Die Information "**Der Apfel ist grün**" wird also mit dem Tripel **Apfel -> ist -> grün** ausgedrückt. Alle Teile eines Triples sind dabei durch weitere Eigenschaften definiert und beschrieben die wiederum in Form von Triples beschrieben sind. Diese vielseitigen Verknüpfungen führen zu einer Netzwerkstruktur, zu einem sogenannten Graphen.

Nachfolgendes Bild aus dem [W3C Primer für RDF](https://www.w3.org/TR/rdf11-primer/) veranschaulicht diese Zusammenhänge:

![](https://www.w3.org/TR/rdf11-primer/example-graph.jpg)

## URI

Eine weitere wichtige Eigenschaft von Linked Data ist, dass alle Teile eines Triples, also Subjekt, Prädikat und Objekt weltweit eindeutig identifizierbar sind. Dafür werden URI (Universal Resource Identifier) eingesetzt. Die URI *xy* beispielsweise ist der weltweit eindeutige Identifier für die Stadt Bern. Typischerweise lassen sich URI **dereferenzieren**, das heisst, ein Request auf die entsprechende URI (bspw. in dem man sie in die Adresszeile eines Browsers eintippt) führt zu einer Website, die Infos zur entsprechenden URI enthält. Im Falle der URI der Stadt Bern wird man auf eine Webpage weitergeleitet, die diverse Informationen zur Stadt Bern enthält.

## Weitere Informationen zu Linked Data

Wer vertieft in das Thema Linked Data einsteigen möchte, dem sei beispielsweise [diese Youtube Playlist](https://www.youtube.com/watch?v=ON0wf0SEPx8&list=PLoOmvuyo5UAfY6jb46jCpMoqb-dbVewxg) empfohlen.

## SPARQL

SPARQL ist eine Query-Sprache für Linked Data Triple Stores. Für eine allgemeine Einführung in SPARQL siehe z.B.: https://jena.apache.org/tutorials/sparql.html

Abfragen (engl. Queries) können entweder direkt über das [SPARQL Web-Interface von Fedlex](https://fedlex.data.admin.ch/de-CH/sparql) eingegeben und ausgeführt werden, oder als HTTP-POST Request an den SPARQL-Endpoint (https://fedlex.data.admin.ch/sparqlendpoint) gesendet werden.

Die letztere Methode erlaubt es, eigene Anwendungen zu bauen, die automatisch aktuelle Daten von Fedlex abfragen können. Für dieses Tutorial verwenden wir diese Methode. Die Abfragen sind jedoch in beiden Fällen identisch.

Um im Tutorial eine neue SPARQL Abfrage zu erstellen, erzeugen Sie eine neue Zelle für Code ("Plus-Zeichen" in der Titelzeile des aktuellen Tabs drücken und im Dropdown "Code" auswählen. Danach können sie mit dem Python Befehl `await query(query_string)` die Abfrage ausführen, welche als Resultat ein Pandas Dataframe zurückgibt, welches sinnvollerweise einer Variable zugewiesen wird. Das Schlüsselwort `await` ist notwendig, weil die Abfrage asynchronen Code enthält. Die Anzeige des Dataframes erfolgt sinnvollerweise mit dem Befehl `display_result(df)`, welcher daführ sorgt, dass die URI im Dataframe als klickbare Links dargestellt werden. Die dreifachen Anführungszeichen des `query_string` ermöglichen, den String über mehrere Zeilen umzubrechen und damit eine übersichtliche Darstellung der Query zu erreichen.

Soll eine Query aus dem Tutorial über das SPARQL Web-Interface ausgeführt werden, kopieren Sie einfach den Teil zwischen den `"""` und fügen sie im [SPARQL Web-Interface](https://ld.admin.ch/sparql) ein.

## Pattern Matching

SPARQL Queries sind Aufträge an den Computer, bestimme Muster (Pattern) in den Daten zu finden (matching). Es können also mit Hilfe von SPARQL Muster vorgegeben werden, und die Datenbank gibt alle Triples zurück, die dieses Muster erfüllen. Einzelne Positionen der Triples können dabei bei einer Abfrage bewusst undefiniert gelassen und mit einer Variable bezeichnet werden. Variablen beginnen mit `?` und werden bei der Abfrage durch alle möglichen Elemente gefüllt, die dieses Pattern erfüllen. Eine ausführlicher Anleitung zum Pattern Matching ist [hier](https://programminghistorian.org/en/lessons/retired/graph-databases-and-SPARQL#rdf-in-brief) zu finden. 

## Prefixes

Um immer wiederkehrende URI kompakter zu notieren, werden sogenannte Prefixes verwendet, im nachfolgenden werden folgende Prefixes benutzt:

```
PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
```

# Tutorial

## Identitäten und Versionen

Unter version.link sind die zentralen Entitäten **Identitäten** und **Versionen**, wobei die Identitäten die eigentlichen Objekte des Interesses darstellen. Hier bspw. die politischen Gemeinden der Schweiz. Diese Identitäten basieren auf einer bestimmten Version. Die Version wiederspiegeln also die zeitliche Entwicklung der Identität. Änderung am Objekt des Interesses (bspw. Namensänderung einer Gemeinde) ziehen eine neue Version nach sich, auf der, die Identität dann entsprechend basiert.

### Identität der Gemeinde "Bern"

In [11]:
df = await query("""

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/351> ?p ?o.

}

""", "https://test.lindas.admin.ch/query")

display_result(df)

,p,o
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://schema.ld.admin.ch/Municipality
1,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://schema.ld.admin.ch/PoliticalMunicipality
2,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://version.link/Identity
3,http://schema.org/name,Bern
4,http://schema.org/identifier,351
5,http://schema.org/isPartOf,https://ld.admin.ch/district/246
6,http://schema.org/inDefinedTermSet,https://ld.admin.ch/dimension/municipality
7,http://schema.org/containedInPlace,https://ld.admin.ch/district/246
8,https://version.link/version,https://ld.admin.ch/municipality/version/15029
9,https://version.link/inVersionedIdentitySet,https://ld.admin.ch/fso/register


Die entscheidenden Angaben sind dabei der Typ `vl:Identity`, die Verbindung zur Version, auf der die Identität basiert über den Link `vl:version` zu `https://ld.admin.ch/municipality/version/15029` und die Einordnung in der Hierarchie über `schema:isPartOf` zu `https://ld.admin.ch/district/246`.

### Version der Gemeinde "Bern"

In [7]:
df = await query("""

SELECT * 
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    <https://ld.admin.ch/municipality/version/15029> ?p ?o.

}

""", "https://test.lindas.admin.ch/query")

display_result(df)

,p,o
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://schema.org/DefinedTerm
1,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://version.link/Version
2,http://schema.org/name,Bern
3,http://schema.org/identifier,15029
4,http://schema.org/isPartOf,https://ld.admin.ch/district/version/10288
5,http://schema.org/validFrom,2010-01-01
6,http://www.geonames.org/ontology#featureCode,http://www.geonames.org/ontology#A.ADM3
7,http://schema.org/legalName,Bern
8,http://schema.org/startDate,2010-01-01
9,http://www.w3.org/ns/prov#hadPrimarySource,https://register.ld.admin.ch/agvch/municipalityversion/15029


Die entscheidenden Angaben sind dabei der Typ `vl:Version`, die Verbindung zur Identität, auf der diese basiert über den Link `vl:identity` zu `https://ld.admin.ch/municipality/351` und die Einordnung in der Hierarchie über `schema:isPartOf` zu `https://ld.admin.ch/district/version/10288`.

## Informationen zu Gemeinde Identitäten

Basierend auf den Identitäten können verschiedene Informationen/Statistiken erstellt werden zum aktuellen Zustand der Gemeindelandschaft der Schweiz:

### Anzahl aller Gemeinden

In [12]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Identity.

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

,count
0,3245


Die Suche nach allen Gemeinden über den Typ `admin:PoliticalMunicipality` ist so zu verstehen, dass alle Gemeinden zurückgeben werden, die aktuell existieren und je existiert haben. Gemeinden, die nicht mehr aktuell sind, erhalten zusätzlich ein Typ `vl:Deprecated`, nachdem gefiltert werden kann:

### Anzahl aller nicht mehr aktuellen Gemeinden

In [13]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Identity.
        a vl:Deprecated.

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

,count
0,1151


### Anzahl aller aktuellen Gemeinden

Wenn nur die aktuellen Gemeinden interessieren, dann muss danach gefiltert werden, dass kein `vl:Deprecated` existieren darf:

In [14]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Identity.
    
    FILTER NOT EXISTS {?muni a vl:Deprecated}

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

,count
0,2094


### Anzahl Gemeinden pro Anfangsbuchstabe

In [21]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?char (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/fso/register>
WHERE 
{
    ?muni a admin:PoliticalMunicipality;
        a vl:Identity;
        schema:name ?name.

    BIND(SUBSTR(?name, 1, 1) AS ?char)

} GROUP BY ?char ORDER BY DESC(?count)

""", "https://test.lindas.admin.ch/query")

display_result(df)

,char,count
0,B,333
1,S,324
2,M,262
3,C,238
4,L,235
5,R,186
6,G,176
7,A,151
8,V,144
9,H,137


### Gemeinden mit 'wil' im Namen

In [35]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?name
FROM <https://lindas.admin.ch/fso/register>
WHERE 
{
    ?muni a admin:PoliticalMunicipality;
        a vl:Identity;
        schema:name ?name.
        
    #FILTER(CONTAINS(?name, 'wil') || CONTAINS(?name, 'Wil'))
    FILTER REGEX(?name, 'wil', 'i').

}

""", "https://test.lindas.admin.ch/query")

display_result(df)

,name
0,Auswil
1,Geltwil
2,Boniswil
3,Wattwil
4,Flawil
5,Abtwil
6,Hinwil
7,Iffwil
8,Nottwil
9,Wattenwil


### Anzahl aktueller Gemeinden pro Kanton

In [15]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?canton ?cantonName ?numberOfMunies
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?muni) AS ?numberOfMunies)
        WHERE {

            ?muni a admin:PoliticalMunicipality;
                a vl:Identity;
                schema:isPartOf/schema:isPartOf ?canton.

            MINUS {?muni a vl:Deprecated}

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfMunies)

""", "https://test.lindas.admin.ch/query")

display_result(df)

,canton,cantonName,numberOfMunies
0,https://ld.admin.ch/canton/2,Bern / Berne,337
1,https://ld.admin.ch/canton/22,Vaud,297
2,https://ld.admin.ch/canton/19,Aargau,197
3,https://ld.admin.ch/canton/1,Zürich,160
4,https://ld.admin.ch/canton/23,Valais / Wallis,120
5,https://ld.admin.ch/canton/10,Fribourg / Freiburg,116
6,https://ld.admin.ch/canton/21,Ticino,105
7,https://ld.admin.ch/canton/11,Solothurn,104
8,https://ld.admin.ch/canton/18,Graubünden / Grigioni / Grischun,101
9,https://ld.admin.ch/canton/13,Basel-Landschaft,86


### Durchschnittliche Grösse der aktuellen Gemeinden pro Kanton

In [67]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?cantonName ?muniMeanSize
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.
    
    SERVICE <https://geo.ld.admin.ch/query> {
    
        ?cantonGEO schema:about ?canton;
            purl:hasVersion ?version.
        ?version purl:issued "2023-01-01"^^xsd:date.
        ?version <http://dbpedia.org/property/area> ?area.
    
    }
    
    {
        SELECT ?canton (COUNT(?muni) AS ?numberOfMunies)
        WHERE {

            ?muni a admin:PoliticalMunicipality;
                a vl:Identity;
                schema:isPartOf/schema:isPartOf ?canton.

            MINUS {?muni a vl:Deprecated}

        } GROUP BY ?canton
    
    }
    
    BIND(ROUND(xsd:float(xsd:string(?area))/?numberOfMunies) AS ?muniMeanSize).

    
} ORDER BY DESC(?muniMeanSize)

""", "https://test.lindas.admin.ch/query")

display_result(df)

,cantonName,muniMeanSize
0,Glarus,22844.0
1,Graubünden / Grigioni / Grischun,7035.0
2,Obwalden,7008.0
3,Uri,5666.0
4,Valais / Wallis,4354.0
5,Appenzell Innerrhoden,3450.0
6,Schwyz,3026.0
7,Neuchâtel,2971.0
8,St. Gallen,2741.0
9,Ticino,2678.0


### Anzahl nicht mehr aktueller Gemeinden pro Kanton

In [16]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?cantonName ?numberOfMunies
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?muni) AS ?numberOfMunies)
        WHERE {

            ?muni a admin:PoliticalMunicipality;
                a vl:Identity;
                schema:isPartOf/schema:isPartOf ?canton;
                a vl:Deprecated.

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfMunies)

""", "https://test.lindas.admin.ch/query")

display_result(df)

,cantonName,numberOfMunies
0,Fribourg / Freiburg,189
1,Ticino,167
2,Thurgau,158
3,Graubünden / Grigioni / Grischun,140
4,Vaud,106
5,Bern / Berne,76
6,Valais / Wallis,66
7,Aargau,44
8,Neuchâtel,40
9,Jura,39


Diese Abfrage zeigt, dass unterschiedliche Kantone das Fusionieren von Gemeinden sehr unterschiedlich stark fördern.

### Anzahl aktueller Bezirke pro Kanton 

In [13]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?cantonName ?numberOfDistricts
FROM <https://lindas.admin.ch/fso/register>
WHERE {
    
    ?canton schema:legalName ?cantonName.

    {
        SELECT ?canton (COUNT(?district) AS ?numberOfDistricts)
        WHERE {

            ?district a admin:District;
                a vl:Identity;
                schema:isPartOf ?canton.

            MINUS {?district a vl:Deprecated}

        } GROUP BY ?canton
    
    }
} ORDER BY DESC(?numberOfDistricts)

""", "https://test.lindas.admin.ch/query")

display_result(df)

,cantonName,numberOfDistricts
0,Valais / Wallis,14
1,Zürich,12
2,Aargau,12
3,Graubünden / Grigioni / Grischun,11
4,Vaud,11
5,Bern / Berne,11
6,Solothurn,10
7,St. Gallen,9
8,Ticino,9
9,Fribourg / Freiburg,8
